# 影像分類入門：ViT 與純視覺基礎（2026 版）

> 模組路徑：`05-Multimodal/01-image_classification/vit_image_classification.ipynb`

這是整個 repo 進入**多模態**世界的第一站。在前面的 `01-04` 模組裡，你已經把文字分類做得滾瓜爛熟：`AutoTokenizer` 把句子轉成 `input_ids`，丟進 `AutoModelForSequenceClassification`，再用 `Trainer` + `evaluate` 跑完訓練與評測。

**這份教材的核心訊息只有一句：影像分類的整條管線跟文字分類幾乎一模一樣，只有兩個地方換掉了。**

| 文字分類（你已熟悉） | 影像分類（這份教材） |
| :--- | :--- |
| `AutoTokenizer` | `AutoImageProcessor` |
| `input_ids` / `attention_mask` | `pixel_values` |
| `AutoModelForSequenceClassification` | `AutoModelForImageClassification` |
| `DataCollatorWithPadding` | `DefaultDataCollator` |
| `Trainer` / `TrainingArguments` | **完全相同** |
| `evaluate.load('accuracy')` | **完全相同** |
| `pipeline('text-classification')` | `pipeline('image-classification')` |

認知落差被刻意壓到最小：把 tokenizer 換成 image_processor、把 `input_ids` 換成 `pixel_values`，其餘照舊。

## 學習目標

讀完這份 notebook，你將能夠：

1. 用 `datasets` 載入影像分類資料集，理解 `Image` feature 與 `ClassLabel` 的結構。
2. 使用 `AutoImageProcessor` 做 resize / rescale / normalize，並理解**為何前處理必須與預訓練模型對齊**。
3. 用 `dataset.with_transform()` 做 **lazy 影像前處理**（對照文字版的 `.map(tokenize)`）。
4. 載入 `AutoModelForImageClassification`，正確覆寫 `num_labels` / `id2label` / `label2id`。
5. 用 `Trainer` 微調，並用 `evaluate` 算 accuracy / F1，畫混淆矩陣做錯誤分析。
6. 用 `pipeline('image-classification')` 做推論並可視化 top-k 機率。
7. 建立 **ViT vs CNN（ConvNeXt）** 的心智模型，理解凍結 backbone / 全微調 / LoRA 的取捨。

## 前置知識

- 已完成文字分類微調，理解 `Trainer` 三件套（model + args + datasets）。建議先看：
  - 文字分類微調：[`../../02-Adv-tasks/01-finetune_optimize/01 train opti classification_demo.ipynb`](../../02-Adv-tasks/01-finetune_optimize/01%20train%20opti%20classification_demo.ipynb)
  - Trainer 元件：[`../../01-Component/06Trainer/06 Trainer classification_demo.ipynb`](../../01-Component/06Trainer/06%20Trainer%20classification_demo.ipynb)
  - evaluate 評測：[`../../01-Component/05evaluate/05 evaluate classification_demo.ipynb`](../../01-Component/05evaluate/05%20evaluate%20classification_demo.ipynb)
  - LoRA 微調（本篇遷移學習段落會呼應）：[`../../03-PEFT/01-LoRA/chatbot_lora.ipynb`](../../03-PEFT/01-LoRA/chatbot_lora.ipynb)

## 銜接

本篇是純視覺（影像 → 標籤）。下一步會進到**影像 + 文字**的跨模態世界（CLIP / SigLIP / VLM），那時你會發現 `pixel_values` 與 `input_ids` 開始**同時出現**在同一個 batch 裡。

## 步驟 1：環境與版本鎖定

**WHY**：多模態管線對版本特別敏感。`AutoImageProcessor`、`AutoModelForImageClassification`、`TrainingArguments` 的 `eval_strategy`（4.46 後取代舊的 `evaluation_strategy`）、`save_safetensors` 等都需要夠新的版本。我們在最前面鎖版本，跟全 repo 的 2026 慣例一致。

- `transformers>=4.46`：新版前處理與 Trainer API。
- `datasets>=3.0`：`Image` feature 與 `with_transform` 的穩定行為。
- `torchvision`：影像增強（augmentation）會用到。
- `evaluate>=0.4`：accuracy / f1 指標。

> 註：相比 LLM 微調，影像分類的 VRAM 需求小很多。本篇主模型 `google/vit-base-patch16-224` 在 224x224、batch 16 下約需 6-8GB VRAM，一般筆電 GPU 即可；純推論用 CPU 也跑得動（慢一些）。

In [ ]:
# Version pinning is the first cell across the whole repo (2026 convention).
# Uncomment to install in a fresh environment.
# !pip install -q "transformers>=4.46" "datasets>=3.0" "evaluate>=0.4" \
#     "accelerate>=1.0" torchvision scikit-learn matplotlib

import transformers, datasets, evaluate, torch
import torchvision

print("transformers:", transformers.__version__)  # expect >= 4.46
print("datasets    :", datasets.__version__)       # expect >= 3.0
print("evaluate    :", evaluate.__version__)        # expect >= 0.4
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)

### device / dtype 慣例

**WHY**：全 repo 統一一個習慣 — 用 `bfloat16`（bf16）當運算精度，因為它在 Ampere（RTX 30 系）以後的 GPU 上又快又穩，數值範圍跟 fp32 一樣（不像 fp16 容易 overflow）。影像分類模型不大，不需要 4-bit 量化（那是給 VLM / LLM 用的），但 dtype / device 的判斷邏輯一模一樣，先建立慣例。

In [ ]:
# Pick device and a sensible compute dtype, following the repo-wide convention.
# bf16 on Ampere+ GPUs; fall back to fp32 on CPU / older GPUs.
if torch.cuda.is_available():
    device = "cuda"
    bf16_ok = torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
else:
    device = "cpu"
    bf16_ok = False
    compute_dtype = torch.float32

print(f"device={device}  bf16_supported={bf16_ok}  compute_dtype={compute_dtype}")

# Reproducibility: same seed convention as every training notebook in the repo.
from transformers import set_seed
set_seed(42)

## 步驟 2：載入影像分類資料集

**WHY**：在文字版你用 `load_dataset` 拿到一個帶 `text` 與 `label` 欄位的資料集。影像版**完全一樣的 API**，只是欄位變成 `image`（一個 `Image` feature）與 `label`（一個 `ClassLabel` feature）。

我們用 **beans**（豆葉病害分類，3 類）當主資料集：它小、下載快、類別語意清楚（健康 / 角斑病 / 鏽病），非常適合教學。CIFAR-10 / Food-101 的用法完全相同，只是更大。

> 重點觀察：`dataset.features` 會告訴你每個欄位的型別。`Image` feature 是 `datasets` 的特殊型別 — 它**不會把整張圖讀進記憶體**，而是存路徑 / bytes，等你真正存取時才 decode。這是影像資料集能放進 RAM 的關鍵。

In [ ]:
from datasets import load_dataset

# beans: 3-class bean leaf disease classification. Small and fast for teaching.
# Swap to load_dataset('cifar10') or load_dataset('food101', split='train[:5%]')
# without changing any downstream code.
ds = load_dataset("beans")
print(ds)

# Inspect features: note `image` is an Image feature, `labels` is a ClassLabel.
print("\nfeatures:", ds["train"].features)

**WHY 看一張圖**：影像資料最容易出錯的地方是「我以為的圖」跟「實際的圖」不一樣（通道順序、尺寸、RGB vs 灰階）。先肉眼看一張、確認標籤對應，再進前處理，能省下大量 debug 時間。`ClassLabel.int2str` 幫你把整數標籤翻回人看得懂的名字 — 這就是後面要餵給模型的 `id2label`。

In [ ]:
# Grab the ClassLabel feature so we can map integer ids <-> human names.
label_feature = ds["train"].features["labels"]
labels = label_feature.names
print("labels:", labels)

# Peek at one example. ds[...]['image'] is a PIL.Image, decoded lazily on access.
example = ds["train"][0]
img = example["image"]
print("PIL image:", img.size, img.mode)
print("label id :", example["labels"], "->", label_feature.int2str(example["labels"]))

import matplotlib.pyplot as plt
plt.figure(figsize=(3, 3))
plt.imshow(img)
plt.title(label_feature.int2str(example["labels"]))
plt.axis("off")
plt.show()

## 步驟 3：AutoImageProcessor — 影像界的 tokenizer

**WHY**：這是這份教材最核心的概念替換。在文字裡，tokenizer 必須跟模型成對使用（同一份詞表、同樣的特殊 token），否則 `input_ids` 對不上模型的 embedding 表。

影像裡完全同理：**image_processor 必須跟模型成對使用**。原因是模型在預訓練時，所有輸入圖都被：

1. **resize** 到固定尺寸（ViT-base 是 224x224，因為它要切成 16x16 的 patch）。
2. **rescale**：把像素值從 `[0, 255]` 縮到 `[0, 1]`。
3. **normalize**：減掉 image mean、除以 image std（ImageNet 的統計值）。

如果你用不同的 mean/std 來 normalize，模型看到的數值分布就跟訓練時不同，準確率會慘跌。所以**永遠用 `from_pretrained` 載入跟模型配對的 processor**，不要自己手刻 normalize。

In [ ]:
from transformers import AutoImageProcessor

model_id = "google/vit-base-patch16-224"  # primary 2026 vision baseline

# Same .from_pretrained pattern as AutoTokenizer in the text notebooks.
image_processor = AutoImageProcessor.from_pretrained(model_id)

# These values are baked into the pretrained model. NEVER hardcode your own.
print("size       :", image_processor.size)        # target HxW, e.g. {'height':224,'width':224}
print("image_mean :", image_processor.image_mean)  # per-channel mean (ImageNet stats)
print("image_std  :", image_processor.image_std)   # per-channel std
print("do_rescale :", image_processor.do_rescale)  # 255 -> [0,1]
print("do_normalize:", image_processor.do_normalize)

**對照其他視覺模型的 processor**：不同模型的 `size` 與 mean/std 不一定相同，這正是「processor 要跟模型配對」的鐵證。

- `google/vit-base-patch16-224` → 224x224，ImageNet mean/std。
- `facebook/dinov2-base` → 自監督預訓練，processor 設定不同。
- `microsoft/swinv2-tiny-patch4-window8-256` → **256x256**（不是 224！）。
- `timm/convnext_tiny.fb_in22k` → CNN 架構，但在 HF 一樣用 `AutoImageProcessor`。

你**不需要記**這些數字，`from_pretrained` 會自動拿到正確值 — 這正是要用它的理由。

In [ ]:
# Demonstrate that different models demand different preprocessing.
# (This cell only loads processors, not models -- cheap.)
for mid in [
    "google/vit-base-patch16-224",
    "microsoft/swinv2-tiny-patch4-window8-256",
]:
    p = AutoImageProcessor.from_pretrained(mid)
    print(f"{mid:48s} size={p.size}  mean={[round(m,3) for m in p.image_mean]}")

## 步驟 4：用 with_transform 做 lazy 影像前處理

**WHY — 這一步在文字版你已經做過**：文字版你寫了一個 `tokenize(batch)` 函式，再 `dataset.map(tokenize, batched=True)`，把 `text` 變成 `input_ids`。

影像版有個關鍵差異：**圖片很大，不適合一次 `.map` 全部前處理後存起來**（會把幾 GB 的 tensor 灌進磁碟 / 記憶體）。所以我們改用 `dataset.with_transform(fn)` — 它是 **lazy 的**：只有當 `Trainer` 真正抓某個 batch 時，才對那幾張圖做 resize/normalize。這對影像是標準作法。

心智對照：

| 文字 | 影像 |
| :--- | :--- |
| `def tokenize(b): return tokenizer(b['text'])` | `def transform(b): ... image_processor(images)` |
| `ds.map(tokenize, batched=True)`（eager，存起來） | `ds.with_transform(transform)`（lazy，現算） |
| 產出 `input_ids` | 產出 `pixel_values` |

我們也順手加一點訓練時的資料增強（random crop / flip），驗證時不增強 — 這跟文字 augmentation 的精神一樣，只是手法不同。

In [ ]:
from torchvision.transforms import (
    Compose, Normalize, RandomResizedCrop, RandomHorizontalFlip,
    Resize, CenterCrop, ToTensor,
)

# Pull the target size and normalization stats FROM the processor (do not invent).
if isinstance(image_processor.size, dict) and "height" in image_processor.size:
    crop_size = (image_processor.size["height"], image_processor.size["width"])
else:
    # Some processors expose {'shortest_edge': N}
    edge = image_processor.size.get("shortest_edge", 224)
    crop_size = (edge, edge)

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)

# Train: light augmentation. Eval: deterministic resize + center crop.
train_tf = Compose([
    RandomResizedCrop(crop_size),
    RandomHorizontalFlip(),
    ToTensor(),      # PIL [0,255] -> float tensor [0,1] (the 'rescale' step)
    normalize,       # ImageNet mean/std, matched to the model
])
eval_tf = Compose([
    Resize(crop_size),
    CenterCrop(crop_size),
    ToTensor(),
    normalize,
])

In [ ]:
# Transform functions: produce `pixel_values`, the image analogue of `input_ids`.
# Note we keep the integer labels under the key the model expects: `labels`.
def apply_train_tf(batch):
    batch["pixel_values"] = [train_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

def apply_eval_tf(batch):
    batch["pixel_values"] = [eval_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

# with_transform is LAZY: nothing is computed until a batch is fetched.
train_ds = ds["train"].with_transform(apply_train_tf)
val_ds   = ds["validation"].with_transform(apply_eval_tf)
test_ds  = ds["test"].with_transform(apply_eval_tf)

# Sanity check one transformed example: a CxHxW float tensor.
sample = train_ds[0]
print("pixel_values shape:", sample["pixel_values"].shape)  # e.g. torch.Size([3, 224, 224])
print("dtype             :", sample["pixel_values"].dtype)
print("label             :", sample["labels"])

## 步驟 5：載入 AutoModelForImageClassification

**WHY — 這一步在文字版你已經做過**：文字版你用 `AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2, id2label=..., label2id=...)`。影像版**逐字對應**，只是 class 換成 `AutoModelForImageClassification`。

兩個必須處理的細節：

1. **`num_labels` 覆寫**：`vit-base-patch16-224` 預訓練在 ImageNet（1000 類），但我們的 beans 只有 3 類。所以要把分類頭（classification head）換成 3 個輸出。
2. **`ignore_mismatched_sizes=True`**：因為我們把 1000 類的頭換成 3 類，權重形狀對不上，這個旗標告訴 transformers「丟掉舊的分類頭、隨機初始化新的、其餘 backbone 照搬」。這正是遷移學習的本質。
3. **`id2label` / `label2id`**：讓模型 / pipeline 輸出人看得懂的標籤名，而不是 `LABEL_0`。

In [ ]:
from transformers import AutoModelForImageClassification

# Build the id<->label maps the model config will carry around.
id2label = {i: name for i, name in enumerate(labels)}
label2id = {name: i for i, name in id2label.items()}
print("id2label:", id2label)

model = AutoModelForImageClassification.from_pretrained(
    model_id,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    # Pretrained head is 1000-way (ImageNet); ours is 3-way -> drop & reinit it.
    ignore_mismatched_sizes=True,
    torch_dtype=compute_dtype,  # bf16 on Ampere+, fp32 otherwise
)

# The warning about newly initialized classifier weights is EXPECTED and correct:
# the backbone is transferred, only the small head is trained from scratch.
print("model class:", model.__class__.__name__)

## 步驟 6：Trainer + TrainingArguments + DefaultDataCollator

**WHY — 這一步幾乎跟文字版一模一樣**：唯一的差別是 collator。文字用 `DataCollatorWithPadding`（因為句子長度不一，要 padding）；影像所有圖都被 resize 成同尺寸，**不需要 padding**，所以用 `DefaultDataCollator`，它只是把一批 dict 疊成 tensor。

`TrainingArguments` 沿用全 repo 2026 慣例：

- `bf16=True`（GPU 支援時）：訓練加速。
- `eval_strategy="epoch"`：每個 epoch 評測一次（注意是新名字，不是舊的 `evaluation_strategy`）。
- `save_safetensors=True`：用 safetensors 格式存檔（安全、快）。
- `seed=42`、`warmup_ratio=0.1`、`lr_scheduler_type="cosine"`：可重現、暖身、cosine 衰減。
- `load_best_model_at_end=True` + `metric_for_best_model="accuracy"`：訓練結束自動回到最佳 checkpoint。

In [ ]:
from transformers import DefaultDataCollator

# No padding needed for images -> the simplest collator just stacks tensors.
data_collator = DefaultDataCollator()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="vit-beans",
    remove_unused_columns=False,   # CRITICAL: keep `image` so with_transform can run
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",         # transformers>=4.46 name (was evaluation_strategy)
    save_strategy="epoch",
    save_safetensors=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=10,
    bf16=bf16_ok,                  # bf16 mixed precision on Ampere+
    seed=42,
    report_to="none",
)

### compute_metrics — 評測 accuracy 與 F1

**WHY — 完全照搬文字版**：`evaluate.load('accuracy')` 與 `evaluate.load('f1')` 的用法跟文字分類零差異。`Trainer` 把模型輸出的 logits 與真實標籤丟進 `compute_metrics`，我們取 `argmax` 當預測。多分類的 F1 用 `average='macro'`（每類同權重，對類別不平衡比較公允）。

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels_true)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels_true, average="macro")["f1"],
    }

### 組裝並訓練

**WHY**：`Trainer` 三件套（model + args + datasets）跟文字版站位完全相同。注意我們把 `image_processor` 傳給 `Trainer`（新版參數名 `processing_class`，取代舊的 `tokenizer`），這樣存模型時 processor 也會一起存，推論時不用再煩惱前處理對齊。

> VRAM 警告：`vit-base` + batch 16 約 6-8GB。若 OOM，可：(1) 把 `per_device_train_batch_size` 降到 8 並設 `gradient_accumulation_steps=2`；(2) 換更小的模型 `microsoft/swinv2-tiny-patch4-window8-256` 或 `WinKawaks/vit-tiny-patch16-224`。純 CPU 也能跑，但一個 epoch 會慢很多，建議先用 `ds['train'].select(range(200))` 做煙霧測試。

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=image_processor,  # transformers>=4.46 (replaces tokenizer=)
)

# Run training. (Outputs intentionally not included in this teaching notebook.)
trainer.train()

In [ ]:
# Final evaluation on the held-out test split.
test_metrics = trainer.evaluate(test_ds)
print(test_metrics)

## 步驟 7：混淆矩陣與錯誤分析

**WHY**：一個 accuracy 數字告訴你「對了多少」，卻不告訴你「在哪裡錯」。混淆矩陣讓你看出**哪兩類最容易被搞混** — 在 beans 裡，兩種病害的早期病徵可能長得很像，模型混淆它們是合理的；但若把「健康」誤判成「病害」就值得追查。這種逐類診斷在文字分類裡你也做過（例如哪兩個情緒類別互相混淆），手法相同。

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Get raw predictions on the test set.
pred_output = trainer.predict(test_ds)
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(5, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix (test)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Drill into a few misclassified examples -- the real signal for error analysis.
mis_idx = np.where(y_pred != y_true)[0][:6]
if len(mis_idx) == 0:
    print("No misclassifications on the test set.")
else:
    fig, axes = plt.subplots(1, len(mis_idx), figsize=(3 * len(mis_idx), 3))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, mis_idx):
        ax.imshow(ds["test"][int(i)]["image"])  # original (un-normalized) image
        ax.set_title(f"T:{labels[y_true[i]]}\nP:{labels[y_pred[i]]}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 步驟 8：pipeline 推論 + top-k 機率可視化

**WHY — 這一步在文字版你已經做過**：文字用 `pipeline('text-classification')`，影像用 `pipeline('image-classification')`，API 一致。pipeline 會自動載入模型 + processor，幫你把 PIL 圖 → `pixel_values` → logits → softmax → 排序好的標籤機率，省掉所有手工步驟。

我們把訓練好的模型存檔後再用 pipeline 載入，模擬真實部署流程；同時可視化 top-k，讓「模型有多確定」這件事看得見。

In [ ]:
# Save the fine-tuned model + processor with safetensors (2026 convention).
save_dir = "vit-beans-final"
trainer.save_model(save_dir)             # saves model with safe_serialization by default
image_processor.save_pretrained(save_dir)
print("saved to:", save_dir)

In [ ]:
from transformers import pipeline

# Same pipeline pattern as text-classification, just a different task string.
clf = pipeline(
    "image-classification",
    model=save_dir,
    device=0 if device == "cuda" else -1,
)

sample_img = ds["test"][0]["image"]
results = clf(sample_img, top_k=len(labels))  # list of {label, score}, sorted desc
for r in results:
    print(f"{r['label']:20s} {r['score']:.4f}")

In [ ]:
# Visualize the image alongside its top-k probability bars.
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(8, 3))
ax_img.imshow(sample_img)
ax_img.set_title("input")
ax_img.axis("off")

names = [r["label"] for r in results][::-1]
scores = [r["score"] for r in results][::-1]
ax_bar.barh(names, scores, color="steelblue")
ax_bar.set_xlim(0, 1)
ax_bar.set_title("top-k probability")
for i, s in enumerate(scores):
    ax_bar.text(s + 0.01, i, f"{s:.2f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## 步驟 9：遷移學習策略 — 凍結 vs 全微調 vs LoRA；ViT vs CNN

**WHY**：你剛剛做的是**全微調**（backbone + 新分類頭全部更新）。但這不是唯一選擇，理解三條路的取捨能幫你在資料量 / 算力受限時做對決定。

### 三種遷移學習策略

| 策略 | 做法 | 適用情境 | 代價 |
| :--- | :--- | :--- | :--- |
| **凍結 backbone（linear probe）** | 凍結所有預訓練層，只訓練新分類頭 | 資料少、算力少、特徵已夠通用（如 DINOv2） | 上限較低，但極快極省 |
| **全微調** | 全部參數可訓練（本篇做的） | 資料夠多、領域跟預訓練差異大 | 最耗算力，易過擬合 |
| **LoRA / PEFT** | 凍結原權重，注入低秩 adapter | 想要全微調的效果 + 凍結的省記憶體 | 介於兩者之間，2026 的主流 |

下面示範**凍結 backbone**，只留分類頭可訓練 — 這是你能立即套用的最省方案。LoRA 在影像模型上的細節留到後面的 PEFT 章節（呼應 [`../../03-PEFT/01-LoRA/chatbot_lora.ipynb`](../../03-PEFT/01-LoRA/chatbot_lora.ipynb)，概念完全可遷移到 ViT）。

In [ ]:
# Demonstration: freeze the backbone, train only the classifier head (linear probe).
# Reload a fresh model so we don't mutate the fully-finetuned one above.
frozen_model = AutoModelForImageClassification.from_pretrained(
    model_id,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    torch_dtype=compute_dtype,
)

# Freeze everything, then re-enable just the classification head.
for param in frozen_model.parameters():
    param.requires_grad = False
for param in frozen_model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in frozen_model.parameters())
print(f"trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
# You'd hand `frozen_model` to a Trainer exactly as before -- only the head learns.

### ViT vs CNN（ConvNeXt）心智模型

**WHY**：你需要知道何時選誰。兩者在 HF 都用同一套 `AutoImageProcessor` + `AutoModelForImageClassification` + `Trainer`，**程式碼幾乎不用改**，差別在歸納偏好（inductive bias）：

| | **ViT / Swin（Transformer）** | **ConvNeXt / ResNet（CNN）** |
| :--- | :--- | :--- |
| 看圖方式 | 切成 patch，用 self-attention 看**全域關係** | 卷積核滑動，看**局部紋理**逐層擴大感受野 |
| 歸納偏好 | 弱（平移不變性要自己學） | 強（內建平移不變、局部性） |
| 資料需求 | 大資料才發揮（或靠預訓練） | 小資料也穩 |
| 2026 定位 | 多模態的共通骨幹（CLIP/VLM 都用 ViT） | 純視覺、邊緣部署、強 baseline |

**實務建議**：要接多模態（下一章）就用 ViT 家族，因為 CLIP / SigLIP / 大多數 VLM 的影像端都是 ViT，心智模型可以直接延續。純分類且資料不多時，ConvNeXt 常是更省事的強 baseline。下面示範換成 ConvNeXt 只需改一個字串。

In [ ]:
# Same API, different backbone family. (Loads weights; comment out if avoiding download.)
convnext_id = "facebook/convnext-tiny-224"  # CNN baseline; timm/convnext_tiny.fb_in22k also works
convnext_processor = AutoImageProcessor.from_pretrained(convnext_id)
convnext_model = AutoModelForImageClassification.from_pretrained(
    convnext_id,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
print("swapped to CNN:", convnext_model.__class__.__name__)
# From here, the Trainer / metrics / pipeline code is byte-for-byte identical.

### 補充：自監督 backbone DINOv2

**WHY**：`facebook/dinov2-base` 是**自監督**預訓練（沒用標籤），它的特徵特別適合「凍結 backbone + linear probe」這條路 — 很多任務上，凍結 DINOv2 的特徵再接一個線性頭，就能逼近全微調的效果，省下大量算力。這跟文字界用凍結的 sentence-embedding 接下游頭是同一個思路。載入方式一致，只是它沒有預訓練分類頭，所以分類頭一定是新初始化的。

In [ ]:
# DINOv2: strong self-supervised features, ideal for frozen-backbone linear probing.
# (Loads weights; comment out to skip the download.)
dino_id = "facebook/dinov2-base"
dino_processor = AutoImageProcessor.from_pretrained(dino_id)
dino_model = AutoModelForImageClassification.from_pretrained(
    dino_id,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
print("DINOv2 head will be newly initialized (no pretrained classifier):",
      dino_model.__class__.__name__)

## 步驟 10：小結與通往 CLIP / VLM 的銜接

### 你學到了什麼

整條影像分類管線，跟你熟悉的文字分類**只差兩個替換**：

1. `AutoTokenizer` → `AutoImageProcessor`（前處理：resize / rescale / normalize，必須跟模型配對）。
2. `input_ids` → `pixel_values`（模型的輸入張量）。

其餘 — `Trainer`、`TrainingArguments`、`evaluate`、`pipeline`、混淆矩陣、遷移學習策略 — **全部照搬**。這就是 HuggingFace 抽象的威力：學會一個模態，其他模態的認知成本被壓到最低。

你也補上了 inventory 中缺席的 **`AutoImageProcessor` 抽象**，這是進入所有多模態主題的鑰匙。

### 練習題

1. **換資料集**：把 `load_dataset('beans')` 改成 `load_dataset('cifar10')`（欄位是 `img` / `label`，記得改 transform 的欄位名與 `id2label`）。觀察 10 類比 3 類難多少。
2. **凍結 vs 全微調**：把步驟 9 的 `frozen_model` 真的丟進 `Trainer` 訓練，比較它跟全微調的 accuracy 與訓練時間，量化「省算力」的代價。
3. **架構對打**：用同樣的 args 分別微調 `vit-base`、`swinv2-tiny`、`convnext-tiny`，比較三者的 accuracy / 速度 / VRAM，驗證步驟 9 的心智模型。
4. **前處理對齊實驗**：故意用錯誤的 mean/std（例如全 0.5）normalize，觀察 accuracy 掉多少，親身體會「processor 必須跟模型配對」。
5. **錯誤分析**：把步驟 7 的混淆矩陣換成 normalized 版本（`confusion_matrix(..., normalize='true')`），找出召回率最低的類別並提出改善假設。

### 通往下一份 notebook

本篇是**純視覺**：輸入只有圖（`pixel_values`），輸出是固定的類別。但現實世界的需求往往是「用自然語言描述要找什麼」—— 這就需要**影像與文字共享同一個語意空間**。

下一站進入**跨模態**：

- **CLIP / SigLIP**：影像端是你剛學的 ViT，文字端是你更早學的 Transformer，兩者被訓練到「相符的圖文在向量空間靠近」。你會看到 `pixel_values` 與 `input_ids` **第一次同時出現在一個 batch**，並用它做 zero-shot 分類與圖文檢索。
  - 圖文檢索的向量檢索基礎，可先複習：[`../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`](../../02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb)
- 之後再進到 **VLM（視覺語言模型）**：用 `processor.apply_chat_template` 把 `[{'type':'image'}, {'type':'text', 'text': ...}]` 餵給模型做圖像問答 / 描述，並用 4-bit 量化 + LoRA 微調（呼應 `03-PEFT` 與 `04-kbits-tuning` 的技巧）。

你在這份 notebook 建立的 ViT 心智模型，會在後續每一個多模態主題裡反覆用到。